# Source exploration: Norwegian catch and vessel data (2023)

This notebook assesses the source for a pipeline analysing fish quantities by landing
region, month and species. It documents access, structure, field meanings and initial
data quality before Bronze ingestion.

| Dataset snapshot | Result |
|---|---:|
| Catch year | 2023 |
| CSV size | 1,209,038,213 bytes (1.21 GB) |
| Records | 1,079,929 |
| Columns | 133 |

All fields are read as text to preserve source values. Counts and missing-value rates
cover the full file; examples and key checks use the first 10,000 correctly sized records.
Original Norwegian field names are retained.

## 1. Source and access

| Property | Description |
|---|---|
| Publisher | Fiskeridirektoratet (Norwegian Directorate of Fisheries) |
| Source | [Catch data linked with vessel data](https://www.fiskeridir.no/statistikk-tall-og-analyse/data-og-statistikk-om-yrkesfiske/apne-data-fangstdata-seddel-koblet-med-fartoydata) |
| Selected file | [fangstdata_2023.csv.zip](https://register.fiskeridir.no/uttrekk/fangstdata_2023.csv.zip) |
| Access | Public HTTPS download; extract the annual ZIP |
| Format | Semicolon-separated CSV, UTF-8, double-quote text qualifier |
| Refresh | Current and previous two catch years: nightly except the night leading into Saturday; older years: when needed |
| Licence | [NLOD; attribution to Fiskeridirektoratet required](https://www.fiskeridir.no/statistikk-tall-og-analyse/lisens-for-bruk-av-fiskeridirektoratets-data) |
| Source reviewed | 2026-11-05 |

2023 provides a full catch year outside the regular nightly refresh window in 2026.
Older files may still be revised. Value data are released 12 months after landing.
The local CSV is identified by its SHA-256 below; its original download date and ZIP
checksum were not recorded.

## 2. Record structure and key fields

Definitions from the [official data dictionary](https://www.fiskeridir.no/statistikk-tall-og-analyse/data-og-statistikk-om-yrkesfiske/apne-data-fangstdata-seddel-koblet-med-fartoydata/_/attachment/download/0e33d183-cd58-489f-8842-e788eda69251%3A62639d19d430b5251db1ca7564d1e4d178eb5348/Dokumentasjon_Fangst_Fart%C3%B8y_%C3%85pne_Data.pdf), dated 2023-12-12;
References use PDF page numbers.

One row is a document's product line. Only the current document version is published;
version 99 denotes Directorate corrections (pp. 1, 3-4).

| Source field(s) | Meaning | Pages |
|---|---|---|
| `Dokumentnummer`, `Salgslag (kode)`, `Dokumenttype (kode)` | Document identity needs sales organisation and type; consider year | 3 |
| `Linjenummer` | Product-line number within document | 26 |
| `Fangstår` / `Siste fangstdato` | Catch year / last catch day | 18 |
| `Landingsdato` | Landing completion date; sometimes sales date; year may differ from catch year | 25 |
| `Landingsklokkeslett` / `Landingstidspunkt` | Local completion time / landing date-time | 25 |
| `Dokument salgsdato` / `Dokument versjonstidspunkt` | Signing date / electronic version generation date-time | 4 |
| `Oppdateringstidspunkt` | Latest data update date-time | 34 |
| `Landingskommune`, `Landingsfylke`, `Landingsnasjon` | Landing geography | 6-8 |
| `Art - FDIR`, `Art`, `Art FAO` | Directorate species, consolidated species, FAO species | 26-27 |
| `Produktvekt` | Net product kilograms | 31 |
| `Rundvekt` | Live-weight-equivalent kilograms; product weight times conversion factor; zero for by-products | 32 |
| `Bruttovekt` | Kilograms including water and packaging | 31 |

Landing-note quantities decrease as sales notes are issued. Recent snapshots may
contain temporary double counting. Vessel-link failures can leave vessel attributes
missing (pp. 1-2).

## 3. Read and inspect the file

Place `fangstdata_2023.csv` in the project root and run the cells in order using a
Python notebook kernel. The analysis uses the standard library and streams the full
file to limit memory use. The original file is read without modification.

In [ ]:
import csv
import hashlib
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

working_dir = Path.cwd().resolve()
ROOT = next(
    (p for p in [working_dir, *working_dir.parents] if (p / "docs" / "design.md").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from the project directory.")
SOURCE = ROOT / "fangstdata_2023.csv"
if not SOURCE.is_file():
    raise FileNotFoundError(f"Place the extracted CSV here: {SOURCE}")
SOURCE_URL = "https://register.fiskeridir.no/uttrekk/fangstdata_2023.csv.zip"
SELECTED_CATCH_YEAR = "2023"
SAMPLE_LIMIT = 10_000
ANALYSED_AT = datetime.now(timezone.utc).isoformat()
SOURCE_BYTES = SOURCE.stat().st_size
print(f"File: {SOURCE.name}")
print(f"Snapshot checked (UTC): {ANALYSED_AT}")
print(f"Selected catch year: {SELECTED_CATCH_YEAR}")
print(f"Download URL (reference only): {SOURCE_URL}")
print(f"File size: {SOURCE_BYTES:,} bytes / {SOURCE_BYTES / 1024**3:.3f} GiB")

File: fangstdata_2023.csv
Snapshot checked (UTC): 2026-11-05T19:26:19.045214+00:00
Selected catch year: 2023
Download URL (reference only): https://register.fiskeridir.no/uttrekk/fangstdata_2023.csv.zip
File size: 1,209,038,213 bytes / 1.126 GiB


In [ ]:
with SOURCE.open("rb") as handle:
    probe = handle.read(256 * 1024)
# Exclude a potentially incomplete final line from the detection probe.
probe_text = probe.rsplit(b"\n", 1)[0].decode("utf-8-sig", errors="strict")
dialect = csv.Sniffer().sniff(probe_text, delimiters=";,\t|")
ENCODING = "utf-8-sig"
DELIMITER = dialect.delimiter
print("Format: CSV (text file)")
print(f"Detected delimiter: {DELIMITER!r}")
print(f"Quote character: {dialect.quotechar!r}")
print(f"UTF-8 BOM present: {probe.startswith(bytes.fromhex('efbbbf'))}")
if DELIMITER != ";":
    raise ValueError("Unexpected separator: investigate before continuing.")

sha256 = hashlib.sha256()
with SOURCE.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b""):
        sha256.update(chunk)
SOURCE_SHA256 = sha256.hexdigest()
print(f"SHA-256 (extracted CSV): {SOURCE_SHA256}")

Format: CSV (text file)
Detected delimiter: ';'
Quote character: '"'
UTF-8 BOM present: False
SHA-256 (extracted CSV): e359759f9a190dd71ad77a3e004e83cc66d2339b914b6a32ad64f43c4596b4bc


The expected raw schema is the fixed, ordered list of 133 column names below.
Every field has type **string**; final analytical types are decided during Silver
modelling. The output previews the first 12 fields; the code contains the complete schema.

In [ ]:
EXPECTED_COLUMNS = [
    "Dokumentnummer",
    "Dokumenttype (kode)",
    "Dokumenttype",
    "Dokument versjonsnummer",
    "Dokument salgsdato",
    "Dokument versjonstidspunkt",
    "Salgslag ID",
    "Salgslag (kode)",
    "Salgslag",
    "Mottaker ID",
    "Mottakernasjonalitet (kode)",
    "Mottakernasjonalitet",
    "Mottaksstasjon",
    "Landingskommune (kode)",
    "Landingskommune",
    "Landingsfylke (kode)",
    "Landingsfylke",
    "Landingsnasjon (kode)",
    "Landingsnasjon",
    "Produksjonsanlegg",
    "Produksjonskommune (kode)",
    "Produksjonskommune",
    "Mottakende fart\u00f8y reg.merke",
    "Mottakende fart\u00f8y rkal",
    "Mottakende fart\u00f8ytype (kode)",
    "Mottakende fart.type",
    "Mottakende fart\u00f8ynasj. (kode)",
    "Mottakende fart.nasj",
    "Fisker ID",
    "Fiskerkommune (kode)",
    "Fiskerkommune",
    "Fiskernasjonalitet (kode)",
    "Fiskernasjonalitet",
    "Fart\u00f8y ID",
    "Registreringsmerke (seddel)",
    "Radiokallesignal (seddel)",
    "Fart\u00f8ynavn",
    "Fart\u00f8ytype (kode)",
    "Fart\u00f8ytype",
    "Kvotefart\u00f8y reg.merke",
    "Besetning",
    "Fart\u00f8ykommune (kode)",
    "Fart\u00f8ykommune",
    "Fart\u00f8yfylke (kode)",
    "Fart\u00f8yfylke",
    "Fart\u00f8ynasjonalitet (kode)",
    "Fart\u00f8ynasjonalitet",
    "Fart\u00f8ynasjonalitet gruppe",
    "St\u00f8rste lengde",
    "Lengdegruppe (kode)",
    "Lengdegruppe",
    "Bruttotonnasje 1969",
    "Bruttotonnasje annen",
    "Bygge\u00e5r",
    "Ombyggings\u00e5r",
    "Motorkraft",
    "Motorbygge\u00e5r",
    "Fangst\u00e5r",
    "Siste fangstdato",
    "Kvotetype (kode)",
    "Kvotetype",
    "Redskap (kode)",
    "Redskap",
    "Redskap - gruppe (kode)",
    "Redskap - gruppe",
    "Redskap - hovedgruppe (kode)",
    "Redskap - hovedgruppe",
    "Fangstfelt (kode)",
    "Kyst/hav (kode)",
    "Hovedomr\u00e5de (kode)",
    "Hovedomr\u00e5de",
    "Lon (hovedomr\u00e5de)",
    "Lat (hovedomr\u00e5de)",
    "Lokasjon (kode)",
    "Lon (lokasjon)",
    "Lat (lokasjon)",
    "Sone (kode)",
    "Sone",
    "Omr\u00e5degruppering (kode)",
    "Omr\u00e5degruppering",
    "Hovedomr\u00e5de FAO (kode)",
    "Hovedomr\u00e5de FAO",
    "Nord/s\u00f8r for 62 grader nord",
    "Fangstdagbok (nummer)",
    "Fangstdagbok (turnummer)",
    "Landingsdato",
    "Landingsklokkeslett",
    "Landingsm\u00e5ned (kode)",
    "Landingsm\u00e5ned",
    "Landingstidspunkt",
    "Dellanding (signal)",
    "Neste mottaksstasjon",
    "Forrige mottakstasjon",
    "Linjenummer",
    "Art - FDIR (kode)",
    "Art - FDIR",
    "Art (kode)",
    "Art",
    "Art - gruppe (kode)",
    "Art - gruppe",
    "Art - hovedgruppe (kode)",
    "Art - hovedgruppe",
    "Art FAO (kode)",
    "Art FAO",
    "Produkttilstand (kode)",
    "Produkttilstand",
    "Konserveringsm\u00e5te (kode)",
    "Konserveringsm\u00e5te",
    "Landingsm\u00e5te (kode)",
    "Landingsm\u00e5te",
    "Kvalitet (kode)",
    "Kvalitet",
    "St\u00f8rrelsesgruppering (kode)",
    "Anvendelse (kode)",
    "Anvendelse",
    "Anvendelse hovedgruppe (kode)",
    "Anvendelse hovedgruppe",
    "Antall stykk",
    "Bruttovekt",
    "Produktvekt",
    "Produktvekt over kvote",
    "Rundvekt over kvote",
    "Rundvekt",
    "Enhetspris for kj\u00f8per",
    "Bel\u00f8p for kj\u00f8per",
    "Enhetspris for fisker",
    "Bel\u00f8p for fisker",
    "St\u00f8ttebel\u00f8p",
    "Lagsavgift",
    "Inndradd fangstverdi",
    "Etterbetaling",
    "Fangstverdi",
    "Oppdateringstidspunkt",
]

with SOURCE.open("r", encoding=ENCODING, errors="strict", newline="") as handle:
    reader = csv.reader(handle, delimiter=DELIMITER, quotechar=dialect.quotechar, strict=True)
    columns = next(reader)

if columns != EXPECTED_COLUMNS:
    added = sorted(set(columns) - set(EXPECTED_COLUMNS))
    missing = sorted(set(EXPECTED_COLUMNS) - set(columns))
    raise ValueError(
        f"Header differs from the A1 baseline. Added={added}; missing={missing}; "
        "also inspect column order. Do not reuse the saved findings without review."
    )
if len(set(columns)) != len(columns):
    raise ValueError("Duplicate column names require investigation.")
expected_raw_schema = [(name, "string") for name in EXPECTED_COLUMNS]
print(f"Header matches baseline: {len(expected_raw_schema)} columns")
for position, (name, raw_type) in enumerate(expected_raw_schema[:12], 1):
    print(f"{position:3}. {name:43} {raw_type}")
print("... Complete 133-field schema is defined above.")

Header matches baseline: 133 columns
  1. Dokumentnummer                              string
  2. Dokumenttype (kode)                         string
  3. Dokumenttype                                string
  4. Dokument versjonsnummer                     string
  5. Dokument salgsdato                          string
  6. Dokument versjonstidspunkt                  string
  7. Salgslag ID                                 string
  8. Salgslag (kode)                             string
  9. Salgslag                                    string
 10. Mottaker ID                                 string
 11. Mottakernasjonalitet (kode)                 string
 12. Mottakernasjonalitet                        string
... Complete 133-field schema is defined above.


Three example records show the fields used in the planned analysis. The sample is
selected from the start of the file and is not assumed to represent all records.

In [ ]:
with SOURCE.open("r", encoding=ENCODING, errors="strict", newline="") as handle:
    reader = csv.reader(handle, delimiter=DELIMITER, quotechar=dialect.quotechar, strict=True)
    next(reader)
    sample = []
    for row in reader:
        if len(row) == len(columns):
            sample.append(dict(zip(columns, row, strict=False)))
        if len(sample) >= SAMPLE_LIMIT:
            break

example_fields = [
    "Landingsdato",
    "Landingsfylke",
    "Art",
    "Produktvekt",
    "Rundvekt",
]
print(f"Sample: {len(sample):,} records; all field values are strings.")
for number, record in enumerate(sample[:3], 1):
    print(f"\nRecord {number}")
    for name in example_fields:
        print(f"  {name:20} {record[name]!r}")

Sample: 10,000 records; all field values are strings.

Record 1
  Landingsdato         '16.01.2023'
  Landingsfylke        'Nordland'
  Art                  'Norsk vårgytende sild'
  Produktvekt          '32420'
  Rundvekt             '32420'

Record 2
  Landingsdato         '16.01.2023'
  Landingsfylke        'Nordland'
  Art                  'Norsk vårgytende sild'
  Produktvekt          '32420'
  Rundvekt             '32420'

Record 3
  Landingsdato         '26.01.2023'
  Landingsfylke        'Nordland'
  Art                  'Norsk vårgytende sild'
  Produktvekt          '37415'
  Rundvekt             '37415'


## 4. Initial data quality

The full scan checks UTF-8 decoding, CSV structure, catch-year coverage and missing
fields. **Empty** means a field contains no value; **whitespace** means it contains
only spaces or other whitespace. Both count as missing. Values such as `0` are retained.
Records with an unexpected field count are counted separately and excluded from
field percentages. The output shows the ten columns with the most missing values.

In [ ]:
started = time.perf_counter()
rows_total = 0
rows_valid_width = 0
blank_records = 0
bad_width = 0
bad_width_examples = []
empty_counts = [0] * len(columns)
whitespace_counts = [0] * len(columns)
catch_year_counts = Counter()
document_type_counts = Counter()
year_index = columns.index("Fangstår")
document_type_index = columns.index("Dokumenttype (kode)")

with SOURCE.open("r", encoding=ENCODING, errors="strict", newline="") as handle:
    reader = csv.reader(handle, delimiter=DELIMITER, quotechar=dialect.quotechar, strict=True)
    if next(reader) != EXPECTED_COLUMNS:
        raise ValueError("Header changed since the baseline check.")
    for row in reader:
        rows_total += 1
        if not row:
            blank_records += 1
        if len(row) != len(columns):
            bad_width += 1
            if len(bad_width_examples) < 5:
                bad_width_examples.append({"csv_record": rows_total, "fields": len(row)})
            continue
        rows_valid_width += 1
        catch_year_counts[row[year_index]] += 1
        document_type_counts[row[document_type_index]] += 1
        for i, value in enumerate(row):
            if value == "":
                empty_counts[i] += 1
            elif value.isspace():
                whitespace_counts[i] += 1

SCAN_SECONDS = time.perf_counter() - started
missing_profile = [
    {
        "column": name,
        "empty": empty_counts[i],
        "whitespace_only": whitespace_counts[i],
        "missing": empty_counts[i] + whitespace_counts[i],
        "missing_pct": round(100 * (empty_counts[i] + whitespace_counts[i]) / rows_valid_width, 4)
        if rows_valid_width
        else None,
    }
    for i, name in enumerate(columns)
]
print(f"\nCSV records after header: {rows_total:,}")
print(f"Records with expected field count: {rows_valid_width:,}")
print(
    f"Records with unexpected field count: {bad_width:,} "
    f"(including {blank_records:,} blank records)"
)
print(f"Unexpected-width examples: {bad_width_examples}")
print(f"Catch-year counts (raw values): {dict(catch_year_counts)}")
print(f"Document-type counts (raw codes): {dict(document_type_counts)}")
print("Entire file read with strict UTF-8 decoding and CSV parsing.")
print(f"Full-scan duration: {SCAN_SECONDS:.1f} seconds")
print("\nTop 10 columns by missing count (entire file):")
print(f"{'Column':43} {'Empty':>10} {'Whitespace':>12} {'Missing %':>12}")
for item in sorted(missing_profile, key=lambda p: p["missing"], reverse=True)[:10]:
    percentage = f"{item['missing_pct']:.4f}" if item["missing_pct"] is not None else "N/A"
    print(f"{item['column']:43} {item['empty']:10,} {item['whitespace_only']:12,} {percentage:>12}")


CSV records after header: 1,079,929
Records with expected field count: 1,079,929
Records with unexpected field count: 0 (including 0 blank records)
Unexpected-width examples: []
Catch-year counts (raw values): {'2023': 1079929}
Document-type counts (raw codes): {'0': 1011621, '1': 68308}
Entire file read with strict UTF-8 decoding and CSV parsing.
Full-scan duration: 57.7 seconds

Top 10 columns by missing count (entire file):
Column                                           Empty   Whitespace    Missing %
Mottaker ID                                  1,079,929            0     100.0000
Neste mottaksstasjon                         1,077,137            0      99.7415
Forrige mottakstasjon                        1,075,921            0      99.6289
Mottakende fartøy rkal                       1,075,470            0      99.5871
Mottakende fartøy reg.merke                  1,075,463            0      99.5865
Mottakende fartøytype (kode)                 1,075,463            0      99.5865
M

| Finding | Interpretation |
|---|---|
| All 1,079,929 records have 133 fields and catch year 2023 | File structure and catch-year coverage match the selected snapshot |
| `Mottaker ID`: 100% empty | The [source change log](https://www.fiskeridir.no/statistikk-tall-og-analyse/data-og-statistikk-om-yrkesfiske/apne-data-fangstdata-seddel-koblet-med-fartoydata/endringer-i-fangstdata-koblet-med-fartoydata) states new values stopped being supplied in March 2022 |
| `Landingsfylke`: 17,721 missing (1.64%) | Geographic coverage needs investigation before regional aggregation |
| `Rundvekt`: one missing value | A handling rule is needed if this becomes the main measure |
| `Art (kode)`, `Landingsdato`, `Produktvekt`: no empty or whitespace-only values | Parsing and domain validity still require later profiling |

Sample values also show comma decimals, trailing spaces in month names, and date-only
values in `Landingstidspunkt` and `Oppdateringstidspunkt`. These require explicit
parsing rules. Missing values are not automatically errors or zero quantities.

## 5. Candidate keys

Three candidates are compared on the 10,000-record sample. `Extra rows` counts
occurrences beyond the first for each complete key. Sample uniqueness does not prove
full-file uniqueness; no records are removed.

In [ ]:
key_candidates = {
    "Document number": ("Dokumentnummer",),
    "Document line": (
        "Fangstår",
        "Salgslag (kode)",
        "Dokumenttype (kode)",
        "Dokumentnummer",
        "Linjenummer",
    ),
    "Document line + version": (
        "Fangstår",
        "Salgslag (kode)",
        "Dokumenttype (kode)",
        "Dokumentnummer",
        "Linjenummer",
        "Dokument versjonsnummer",
    ),
}
print(f"Key checks on {len(sample):,} sample records")
print(f"{'Candidate':26} {'Missing':>8} {'Distinct':>9} {'Extra rows':>11}")
for label, key in key_candidates.items():
    complete = [record for record in sample if all(record[c].strip() for c in key)]
    frequencies = Counter(tuple(record[c] for c in key) for record in complete)
    extra = sum(n - 1 for n in frequencies.values())
    print(f"{label:26} {len(sample) - len(complete):8,} {len(frequencies):9,} {extra:11,}")

exact_sample_rows = Counter(tuple(record[c] for c in columns) for record in sample)
print("\nIdentical full rows, extra occurrences:", sum(n - 1 for n in exact_sample_rows.values()))

Key checks on 10,000 sample records
Candidate                   Missing  Distinct  Extra rows
Document number                   0     3,734       6,266
Document line                     0    10,000           0
Document line + version           0    10,000           0

Identical full rows, extra occurrences: 0


**Document line** combines catch year, sales organisation, document type, document
number and line number. It is unique in the sample, while document number alone
repeats. The composite key must be tested on the full dataset before adoption.

## 6. Conclusion

The 2023 snapshot provides a documented annual source with 133 fields and approximately
1.08 million records. It contains the landing date, region, species and weight fields
needed for the planned analysis. Raw fields will be preserved as text during ingestion.

The main issues to resolve during profiling are missing geography, the missing
`Rundvekt` value, date/time formats and full-data key uniqueness. One documented
weight measure must be selected for aggregation.

The next step is Bronze ingestion of the annual archive with source metadata and
checksums. This notebook establishes the source understanding and expected raw schema.